# Vector Retriever

This notebook introduces the `VectorRetriever` for semantic search and connects it to an LLM to build a complete GraphRAG question-answering pipeline.

**Learning Objectives:**
- Use `VectorRetriever` to find semantically similar chunks
- Build a `GraphRAG` pipeline that retrieves context and generates answers
- Inspect the context passed to the LLM

The retrieval pipeline works in three steps: the query is embedded, the vector index returns the most similar chunks, and those chunks are passed to the LLM as context for generating an answer.

In [ ]:
%pip install "neo4j-graphrag[bedrock] @ git+https://github.com/neo4j-partners/neo4j-graphrag-python.git@bedrock-embeddings" -q

In [ ]:
import os

from lib.data_utils import get_embedder, get_llm
from neo4j_graphrag.retrievers import VectorRetriever
from neo4j_graphrag.generation import GraphRAG
from neo4j import GraphDatabase
from dotenv import load_dotenv

# Load configuration
load_dotenv('../CONFIG.txt')

NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
driver.verify_connectivity()

embedder = get_embedder()
llm = get_llm()

print(f'Connected to Neo4j!')
print(f'LLM: {llm.model_id}')
print(f'Embedder: {embedder.model_id}')

## Initialize Vector Retriever

The `VectorRetriever` wraps the vector index and handles embedding the query, running the similarity search, and formatting the results. The `return_properties` parameter controls which node properties are included in the retrieval output.

In [ ]:
vector_retriever = VectorRetriever(
    driver=driver,
    index_name='chunkEmbeddings',
    embedder=embedder,
    return_properties=['text']
)

print('Vector retriever initialized!')

## Search for Relevant Chunks

Use the retriever to find the 5 chunks most semantically similar to a query. Each result includes a similarity score and the chunk text.

In [ ]:
query = "What are Apple's main products?"
result = vector_retriever.search(query_text=query, top_k=5)

print(f'Query: "{query}"')
print(f'Results returned: {len(result.items)}\n')

for i, item in enumerate(result.items, 1):
    score = item.metadata.get('score', 0.0)
    content_preview = str(item.content)[:150]
    print(f'{i}. Score: {score:.4f}')
    print(f'   {content_preview}...\n')

## Build GraphRAG Pipeline

The `GraphRAG` class connects the retriever to the LLM. When you call `search()`, it retrieves relevant chunks, assembles them into a context prompt, and sends that prompt to the LLM for answer generation.

Setting `return_context=True` lets you inspect the exact chunks the LLM received.

In [ ]:
rag = GraphRAG(llm=llm, retriever=vector_retriever)

query = "What are the key risk factors mentioned in Apple's 10-K filing?"
response = rag.search(query, retriever_config={'top_k': 5}, return_context=True)

print(f'Query: "{query}"')
print(f'Chunks retrieved: {len(response.retriever_result.items)}\n')
print('Answer:')
print(response.answer)

print('\n\n=== Retrieved Context ===')
for i, item in enumerate(response.retriever_result.items, 1):
    score = item.metadata.get('score', 0.0)
    content_str = str(item.content)
    preview = content_str[:200] + '...' if len(content_str) > 200 else content_str
    print(f'\n[{i}] Score: {score:.4f}')
    print(f'    {preview}')

## Try Different Queries

Experiment with different questions to see how semantic search finds relevant content even when the exact words differ from the source text.

In [ ]:
query = "What financial metrics indicate Apple's performance?"

response = rag.search(query, retriever_config={'top_k': 3})

print(f'Query: "{query}"\n')
print('Answer:')
print(response.answer)

## Summary

You built a complete semantic question-answering pipeline:

1. **VectorRetriever** embeds the query and finds the most similar chunks by cosine similarity
2. **GraphRAG** passes those chunks to the LLM as context for answer generation
3. **Context inspection** shows exactly what information the LLM used

The `VectorRetriever` works well for general semantic queries, but it returns only the matched chunks without any surrounding context from the graph. In the next notebook, the `VectorCypherRetriever` adds graph traversal to pull in neighboring chunks and document metadata alongside each vector match.

---

**Next:** [VectorCypher Retriever](03_vector_cypher_retriever.ipynb)

In [ ]:
# Cleanup
driver.close()
print('Connection closed.')